*0.1 Python for GenAI*

# validators

**The situation.** Your invoicing service has one rule everyone agrees on: *total must equal subtotal plus tax*. The rule is coded in the web form, again in the CSV import script, and again in the path where the model extracts invoices from PDFs. Someone fixes a rounding bug in the form. A month later the import script and the form disagree by one cent on 3,000 invoices, and finance asks which number is right.

**The fix: put the rule on the data itself.** A *validator* is a check attached to the Pydantic model. Wherever an `Invoice` is created — form, import, model output — the same check runs. One place, one answer.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Two kinds of check.** A field validator looks at one field (clean it up, or reject it). A model validator looks at the whole object after every field is set, so it can compare fields to each other.

In [2]:
from pydantic import BaseModel, Field, field_validator, model_validator


class Invoice(BaseModel):
    invoice_id: str = Field(pattern=r"^INV-\d{4}$")  # must look like INV-0042
    customer: str
    subtotal: float = Field(ge=0)
    tax_rate: float = Field(ge=0, le=0.5)
    total: float

    @field_validator("customer")
    @classmethod
    def clean_customer(cls, value: str) -> str:
        # One field: strip spaces, fix the casing, reject blanks.
        if not value.strip():
            raise ValueError("customer must not be blank")
        return value.strip().title()

    @model_validator(mode="after")
    def total_must_match(self) -> "Invoice":
        # The whole object: total must equal subtotal × (1 + tax_rate), to the cent.
        expected = round(self.subtotal * (1 + self.tax_rate), 2)
        if abs(self.total - expected) > 0.01:
            raise ValueError(
                f"total {self.total} does not match subtotal × (1 + tax_rate) = {expected}"
            )
        return self


invoice = Invoice(
    invoice_id="INV-0042", customer="  acme corp ", subtotal=100, tax_rate=0.18, total=118
)
print("accepted:", invoice.customer, "| total", invoice.total)
assert invoice.customer == "Acme Corp"

accepted: Acme Corp | total 118.0


**Reading the output.** The customer name came in as `"  acme corp "` and was stored as `"Acme Corp"` — the field validator cleaned it. The total 118 matched 100 × 1.18, so the model validator let it through.

**Now three invoices that break a rule each.** In real life these come from the form, the import and the model. Here they come from a list, but the check is the same code.

In [3]:
from pydantic import ValidationError

bad_invoices = {
    "wrong id format": {
        "invoice_id": "42",
        "customer": "Acme",
        "subtotal": 100,
        "tax_rate": 0.18,
        "total": 118,
    },
    "blank customer": {
        "invoice_id": "INV-0001",
        "customer": "   ",
        "subtotal": 1,
        "tax_rate": 0,
        "total": 1,
    },
    "total does not add up": {
        "invoice_id": "INV-0001",
        "customer": "Acme",
        "subtotal": 100,
        "tax_rate": 0.18,
        "total": 999,
    },
}
rejected = 0
for label, data in bad_invoices.items():
    try:
        Invoice.model_validate(data)
    except ValidationError as error:
        rejected += 1
        print(f"{label:<24} → {error.errors()[0]['msg']}")
assert rejected == 3

wrong id format          → String should match pattern '^INV-\d{4}$'
blank customer           → Value error, customer must not be blank
total does not add up    → Value error, total 999.0 does not match subtotal × (1 + tax_rate) = 118.0


**Reading the output.** Each bad invoice was rejected with a message that says exactly which rule it broke. The message is the one you wrote in the validator, so it can be sent straight back to the form, or logged next to the imported row.

**The rule to remember.** If a rule must be true everywhere the data exists, it belongs on the model, not in the code that happens to create the data today.

| Use it when | Don't when | Instead use |
|---|---|---|
| a business rule must hold everywhere the data appears | the check needs a database or the network (is this customer id real?) | a service-layer check for lookups; a database constraint as the last line of defence |

**Watch out**
- Validators run every time an object is created. A database call inside one turns 10,000 imports into 10,000 queries.
- Write the message for the person who will read it: "total 999 does not match 118" beats "invalid".
- Cleaning is fine (spaces, casing). Changing meaning is not — never rewrite a product code inside a validator.